Github: https://github.com/mproszewska/B-XAIC <br />
Paper: https://arxiv.org/abs/2505.22252 <br />
Datensatz: https://huggingface.co/datasets/mproszewska/B-XAIC/tree/main <br />

## Installing requirements

In [47]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


## Imports

In [48]:
from datasets import load_dataset

## Load data from Huggingface

In [ ]:
ds = load_dataset("mproszewska/B-XAIC", split="train")  # von huggingface, man muss zuerst huggingface-cli runterladen und sich anmelden
df = ds.to_pandas()
path_to_local_explamnations_sdf = "explanations.sdf"  # huggingface lädt die iwie nicht mit, muss man lokal runterladen
print(df.shape)
print(df.columns.tolist())

(50000, 15)
['Unnamed: 0', 'ChEMBL ID', 'smiles', 'rings-count', 'rings-max', 'X', 'P', 'B', 'indole', 'PAINS', 'split_0', 'split_1', 'split_2', 'split_3', 'split_4']


In [ ]:
# aus https://github.com/mproszewska/B-XAIC/blob/main/dataset.py
TASKS = ["B", "P", "X", "indole", "PAINS", "rings-count", "rings-max"]
SYMBOLS = ["C", "N", "O", "F", "Cl", "Br", "P", "S", "B", "I", "Unk"]
# Mapping: Task-Name => Property-Name für die explanations.sdf
TASK_TO_PROP = {
    "B":           "B",
    "P":           "P",
    "X":           "X",
    "indole":      "indole",
    "PAINS":       "pains",
    "rings-count": "rings",
    "rings-max":   "largest_rings",
}

| Was muss das GNN lernen? | Anforderung | Schwierigkeit |
| :--- | :--- | :--- |
| Bor (B)| einzelnes Atom finden | sehr leicht |
| Phosphor (P) | einzelnes Atom finden | sehr leicht |
| Halogene (X) | eines von mehreren Atomen finden | leicht |
| Indol (I)| konkrete Struktur erkennen | mittel |
| PAINS | viele komplexe Strukturen erkennen | schwer |
| rings-count | The model should predict if a molecule contains more than four rings. | sehr schwer |
| rings-max | Detecting large rings with more than six atoms. | sehr schwer |

In [51]:
print(f"Dataset: {len(df):,} molecules, {df['smiles'].nunique():,} unique SMILES")
print(f"Avg. atoms per molecule (proxy via SMILES length): {df['smiles'].str.len().mean():.1f} chars\n")

print("Task label distribution (% positiv))") # wie auf S.5 im paper
for task in TASKS:
    bar = "█" * int(df[task].mean() * 30)
    print(f"  {task:12s} {df[task].mean():5.1%}  {bar}")

print(f"\nSplit (split_0):")
print(df["split_0"].value_counts().to_string())

Dataset: 50,000 molecules, 50,000 unique SMILES
Avg. atoms per molecule (proxy via SMILES length): 66.9 chars

Task label distribution (% positiv))
  B             2.2%  
  P            13.2%  ███
  X            55.9%  ████████████████
  indole       36.8%  ███████████
  PAINS        32.9%  █████████
  rings-count  30.1%  █████████
  rings-max     5.7%  █

Split (split_0):
split_0
train    40000
valid     5000
test      5000


In [ ]:
# Helper-funktionen aus dataset.py
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from rdkit.Chem import BondType

def atom_label(atom):
    sym = atom.GetSymbol()
    return SYMBOLS.index(sym) if sym in SYMBOLS else len(SYMBOLS) - 1

def bond_type_to_int(bond_type):
    if bond_type == BondType.SINGLE:   return 0
    elif bond_type == BondType.DOUBLE: return 1
    elif bond_type == BondType.TRIPLE: return 2
    elif bond_type == BondType.AROMATIC: return 3
    else: return -1

def smiles_to_graph(mol):
    x = F.one_hot(
        torch.tensor([atom_label(atom) for atom in mol.GetAtoms()], dtype=torch.long),
        len(SYMBOLS),
    ).float()

    row, col, edge_labels = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        row += [i, j]
        col += [j, i]
        edge_type = bond_type_to_int(bond.GetBondType())
        edge_labels += [edge_type, edge_type]

    edge_index = torch.tensor([row, col], dtype=torch.long)
    edge_attr  = torch.tensor(edge_labels, dtype=torch.long).view(-1, 1)
    return x, edge_index, edge_attr

In [ ]:
# aus dataset.py
import pandas as pd
from rdkit import Chem
from tqdm import tqdm

TASK = "indole"

property = TASK_TO_PROP[TASK]

splits = df[[f"split_{i}" for i in range(5)]]
ys     = torch.tensor(df[TASK].tolist()).unsqueeze(1)

dataset = []
skipped = 0

with Chem.SDMolSupplier(path_to_local_explamnations_sdf, sanitize=False) as suppl:
    for y, mol in tqdm(zip(ys, suppl), total=len(ys)):
        if mol is None:
            skipped += 1
            continue

        x, edge_index, edge_attr = smiles_to_graph(mol)

        p = mol.GetProp(property)
        expl_node_mask = torch.zeros(len(x), dtype=torch.bool)
        if p != "":
            nodes = torch.tensor([int(n) for n in p.split(",")], dtype=torch.long)
            expl_node_mask[nodes] = True

        if property not in ["B", "P", "X"]:
            p = mol.GetProp(f"{property}_edge")
            expl_edge_mask = torch.zeros(edge_index.shape[1], dtype=torch.bool)
            if p != "":
                edges = {(int(e1), int(e2)) for e1, e2 in [e.split("#") for e in p.split(",")]}
                for i in range(edge_index.shape[1]):
                    e1, e2 = edge_index[0, i].item(), edge_index[1, i].item()
                    expl_edge_mask[i] = (e1, e2) in edges or (e2, e1) in edges
        else:
            expl_edge_mask = None

        dataset.append(Data(
            x=x, edge_index=edge_index, edge_attr=edge_attr, y=y,
            expl_node_mask=expl_node_mask, expl_edge_mask=expl_edge_mask,
        ))

print(f"Graphen gesamt : {len(dataset)}")

100%|██████████| 50000/50000 [00:49<00:00, 1014.64it/s]

Graphen gesamt : 50000


In [ ]:
#Splits und DataLoader aus dataset.py(get_splits), (train_model.py) 
from torch.utils.data import Subset, WeightedRandomSampler
from torch_geometric.loader import DataLoader

SPLIT = 0
BATCH = 64

idx = torch.arange(len(splits))
split_col = splits[f"split_{SPLIT}"]
train_idx = idx[split_col == "train"]
val_idx   = idx[split_col == "valid"]
test_idx  = idx[split_col == "test"]

train_set = Subset(dataset, train_idx)
val_set   = Subset(dataset, val_idx)
test_set  = Subset(dataset, test_idx)

y_train        = torch.cat([dataset[i].y for i in train_idx])
class_counts   = torch.bincount(y_train)
sample_weights = (1.0 / class_counts.float())[y_train]
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

loader_train = DataLoader(train_set, batch_size=BATCH, sampler=sampler)
loader_val   = DataLoader(val_set,   batch_size=BATCH, shuffle=False)
loader_test  = DataLoader(test_set,  batch_size=BATCH, shuffle=False)

print(f"Task: {TASK}  |  Split: {SPLIT}")
print(f"Train: {len(train_set)}  Val: {len(val_set)}  Test: {len(test_set)}")

Task: indole  |  Split: 0
Train: 40000  Val: 5000  Test: 5000


## GCN, GAT, GIN definition

In [55]:
# aus train_model.py (get_model)
from torch.nn import Linear
from torch_geometric.nn import Sequential, GCN, GAT, GIN, global_add_pool

def get_model(model_type, num_node_features, num_classes,
              hidden_dim=32, num_layers=3, linear_dim=32):

    if model_type == "GCN":
        model = Sequential("x, edge_index, batch", [
            (GCN(num_node_features, hidden_dim,
                 num_layers=num_layers, out_channels=linear_dim),
             "x, edge_index -> x"),
            (global_add_pool, "x, batch -> x"),
            Linear(linear_dim, num_classes),
        ])

    elif model_type == "GAT":
        model = Sequential("x, edge_index, batch", [
            (GAT(num_node_features, hidden_dim,
                 num_layers=num_layers, out_channels=linear_dim,
                 act="elu", dropout=0.6),
             "x, edge_index -> x"),
            (global_add_pool, "x, batch -> x"),
            Linear(linear_dim, num_classes),
        ])

    elif model_type == "GIN":
        model = Sequential("x, edge_index, batch", [
            (GIN(num_node_features, hidden_dim,
                 num_layers=num_layers, out_channels=linear_dim,
                 norm="batch_norm"),
             "x, edge_index -> x"),
            (global_add_pool, "x, batch -> x"),
            Linear(linear_dim, num_classes),
        ])

    return model

## Training

In [ ]:
# wie in train_model.py
def train(model, optimizer, dataloader, device):
    return


def test(model, dataloader, device):
    return

def main():
    return

## Explainations